In [1]:
# Download the shared project code
!wget -q -O sepsis.py https://raw.githubusercontent.com/kna20/early-sepsis-prediction/main/sepsis.py

# Import libraries
import sepsis as s
import pandas as pd
import numpy as np

# Verify everything loaded
print("sepsis.py imported successfully!")
print("Random seed:", s.SEED)
print("Original features:", len(s.ALL_FEATURES))

sepsis.py imported successfully!
Random seed: 42
Original features: 40


In [2]:
# Load both hospitals
data_A = s.load_hospital("A")
data_B = s.load_hospital("B")

# Check the datasets
print("Hospital A:", data_A.shape)
print("Hospital B:", data_B.shape)

print("Hospital A patients:", data_A["PatientID"].nunique())
print("Hospital B patients:", data_B["PatientID"].nunique())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hospital A: (790215, 43)
Hospital B: (761995, 43)
Hospital A patients: 20336
Hospital B patients: 20000


In [3]:
dev_ids, test_ids = s.split_patients(data_A)

print("Training patients:", len(dev_ids))
print("Test patients:", len(test_ids))
print("Overlap:", len(set(dev_ids) & set(test_ids)))

Training patients: 16268
Test patients: 4068
Overlap: 0


In [4]:
# Prepare the development and test patients separately
dev_A = s.prepare_dev(data_A, dev_ids)
test_A = s.prepare_dev(data_A, test_ids)

print("Development rows:", len(dev_A["labels"]))
print("Test rows:", len(test_A["labels"]))
print("Development features:", dev_A["filled"].shape[1])
print("Test features:", test_A["filled"].shape[1])

Development rows: 631120
Test rows: 159095
Development features: 40
Test features: 40


In [5]:
# Prepare all Hospital B patients for testing
test_B = s.prepare_dev(data_B, data_B["PatientID"].unique())

print("Hospital B test rows:", len(test_B["labels"]))
print("Hospital B features:", test_B["filled"].shape[1])

Hospital B test rows: 761995
Hospital B features: 40


In [6]:
from sklearn.preprocessing import StandardScaler

# Original baseline features
cols = s.ALL_FEATURES

# Drop features that were >=99% missing in training
missing_pct = dev_A["missing"][cols].mean() * 100
keep = [c for c in cols if missing_pct[c] < 99]

# Calculate imputation values using Hospital A development only
fill_values = (
    dev_A["filled"][keep]
    .drop(columns=s.UNIT_COLS, errors="ignore")
    .median()
    .to_dict()
)
fill_values.update({c: -1 for c in s.UNIT_COLS if c in keep})

# Apply the SAME imputation values to all three datasets
X_train = dev_A["filled"][keep].fillna(fill_values).copy()
X_test_A = test_A["filled"][keep].fillna(fill_values).copy()
X_test_B = test_B["filled"][keep].fillna(fill_values).copy()

# Standardize using training statistics only
scale_cols = [c for c in keep if c not in s.CATEGORICAL]
scaler = StandardScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_A[scale_cols] = scaler.transform(X_test_A[scale_cols])
X_test_B[scale_cols] = scaler.transform(X_test_B[scale_cols])

print("Features retained:", len(keep))
print("Dropped features:", [c for c in cols if c not in keep])
print("Training shape:", X_train.shape)
print("Hospital A test shape:", X_test_A.shape)
print("Hospital B test shape:", X_test_B.shape)
print("Remaining missing values:",
      X_train.isna().sum().sum()
      + X_test_A.isna().sum().sum()
      + X_test_B.isna().sum().sum())

Features retained: 36
Dropped features: ['EtCO2', 'Bilirubin_direct', 'TroponinI', 'Fibrinogen']
Training shape: (631120, 36)
Hospital A test shape: (159095, 36)
Hospital B test shape: (761995, 36)
Remaining missing values: 0


In [7]:
from sklearn.linear_model import LogisticRegression

# Same settings as sepsis.py
model_weighted = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

# Train on Hospital A development patients only
model_weighted.fit(X_train, dev_A["labels"])

# Predict on both test datasets
prob_A = model_weighted.predict_proba(X_test_A)[:, 1]
prob_B = model_weighted.predict_proba(X_test_B)[:, 1]

print("Model trained successfully!")
print("Hospital A predictions:", len(prob_A))
print("Hospital B predictions:", len(prob_B))

Model trained successfully!
Hospital A predictions: 159095
Hospital B predictions: 761995


In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_test(test, probabilities, threshold=0.55):
    y = test["labels"]
    predictions = (probabilities >= threshold).astype(int)

    utility = s.make_utility_scorer(
        test["patients"], y, test["hours"]
    )

    return {
        "AUROC": roc_auc_score(y, probabilities),
        "AUPRC": average_precision_score(y, probabilities),
        **s.threshold_metrics(y, predictions),
        "Clinical utility": utility(predictions)
    }

# Evaluate the same trained model on both hospitals
results = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(test_A, prob_A),
    "Hospital B": evaluate_test(test_B, prob_B)
}).T

print(results.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7591  0.0793  0.1153     0.0640  0.5771  0.1856            0.3419
Hospital B        0.7232  0.0548  0.0868     0.0477  0.4822  0.1382            0.2126


In [10]:
# Train the unweighted baseline
model_unweighted = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight=None
)

model_unweighted.fit(X_train, dev_A["labels"])

# Predict on both hospitals
prob_A_unweighted = model_unweighted.predict_proba(X_test_A)[:, 1]
prob_B_unweighted = model_unweighted.predict_proba(X_test_B)[:, 1]

# Evaluate using its original threshold
results_unweighted = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_unweighted, threshold=0.10
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_unweighted, threshold=0.10
    )
}).T

print(results_unweighted.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7590  0.0803  0.1230     0.1359  0.1124  0.0157            0.0954
Hospital B        0.7217  0.0553  0.1143     0.1108  0.1181  0.0136            0.0951


In [11]:
kept_clinical = [c for c in s.CLINICAL if c in keep]
flag_cols = [c + "_measured" for c in kept_clinical]

X_train_flags = pd.concat(
    [X_train, dev_A["flags"][flag_cols]], axis=1
)
X_test_A_flags = pd.concat(
    [X_test_A, test_A["flags"][flag_cols]], axis=1
)
X_test_B_flags = pd.concat(
    [X_test_B, test_B["flags"][flag_cols]], axis=1
)

print("Flags added:", len(flag_cols))
print("Training shape:", X_train_flags.shape)
print("Hospital A shape:", X_test_A_flags.shape)
print("Hospital B shape:", X_test_B_flags.shape)

Flags added: 30
Training shape: (631120, 66)
Hospital A shape: (159095, 66)
Hospital B shape: (761995, 66)


In [12]:
# Weighted Logistic Regression with measurement flags
model_flags = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

# Train on Hospital A development data
model_flags.fit(X_train_flags, dev_A["labels"])

# Predict on both hospitals
prob_A_flags = model_flags.predict_proba(X_test_A_flags)[:, 1]
prob_B_flags = model_flags.predict_proba(X_test_B_flags)[:, 1]

# Evaluate using the original cutoff
results_flags = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_flags, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_flags, threshold=0.55
    )
}).T

print(results_flags.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7747  0.0844  0.1166     0.0647  0.5884  0.1873            0.3540
Hospital B        0.7139  0.0516  0.0818     0.0449  0.4545  0.1386            0.1796


In [13]:
# Same 30 clinical measurements used for the flags
since_cols = [c + "_hours_since" for c in kept_clinical]

# Standardize hours-since features using Hospital A development only
scaler_since = StandardScaler()

train_since = scaler_since.fit_transform(dev_A["since"][since_cols])
test_A_since = scaler_since.transform(test_A["since"][since_cols])
test_B_since = scaler_since.transform(test_B["since"][since_cols])

# Add them to the existing baseline + flags inputs
X_train_flags_since = X_train_flags.copy()
X_test_A_flags_since = X_test_A_flags.copy()
X_test_B_flags_since = X_test_B_flags.copy()

X_train_flags_since[since_cols] = train_since
X_test_A_flags_since[since_cols] = test_A_since
X_test_B_flags_since[since_cols] = test_B_since

print("Training:", X_train_flags_since.shape)
print("Hospital A:", X_test_A_flags_since.shape)
print("Hospital B:", X_test_B_flags_since.shape)

Training: (631120, 96)
Hospital A: (159095, 96)
Hospital B: (761995, 96)


In [14]:
# Weighted Logistic Regression with flags + hours since
model_flags_since = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

# Train on Hospital A development patients only
model_flags_since.fit(X_train_flags_since, dev_A["labels"])

# Predict on both test datasets
prob_A_flags_since = model_flags_since.predict_proba(X_test_A_flags_since)[:, 1]
prob_B_flags_since = model_flags_since.predict_proba(X_test_B_flags_since)[:, 1]

# Evaluate at the original threshold (0.55)
results_flags_since = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_flags_since, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_flags_since, threshold=0.55
    )
}).T

print(results_flags_since.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7956  0.0902  0.1216     0.0672  0.6346  0.1937            0.3892
Hospital B        0.7459  0.0575  0.0970     0.0545  0.4381  0.1090            0.2251


In [15]:
# Keep only the clinical features that survived preprocessing
clinical_cols = [c for c in s.CLINICAL if c in keep]

# Same weighted Logistic Regression
model_clinical = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_clinical.fit(
    X_train[clinical_cols],
    dev_A["labels"]
)

prob_A_clinical = model_clinical.predict_proba(
    X_test_A[clinical_cols]
)[:, 1]

prob_B_clinical = model_clinical.predict_proba(
    X_test_B[clinical_cols]
)[:, 1]

results_clinical = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_clinical, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_clinical, threshold=0.55
    )
}).T

print("Clinical features:", len(clinical_cols))
print(results_clinical.round(4).to_string())

Clinical features: 30
                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.6769  0.0425  0.0809     0.0442  0.4737  0.2252            0.1788
Hospital B        0.6828  0.0378  0.0579     0.0308  0.4992  0.2258            0.0410


In [16]:
demo_cols = s.DEMOGRAPHICS

model_demo = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_demo.fit(X_train[demo_cols], dev_A["labels"])

prob_A_demo = model_demo.predict_proba(
    X_test_A[demo_cols]
)[:, 1]

prob_B_demo = model_demo.predict_proba(
    X_test_B[demo_cols]
)[:, 1]

results_demo = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_demo, threshold=0.60
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_demo, threshold=0.60
    )
}).T

print("Features:", demo_cols)
print(results_demo.round(4).to_string())

Features: ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS']
                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7283  0.0750  0.1463     0.0903  0.3850  0.0853            0.2812
Hospital B        0.6366  0.0404  0.0942     0.0552  0.3193  0.0784            0.1711


In [17]:
no_iculos_cols = [c for c in keep if c != "ICULOS"]

model_no_iculos = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_no_iculos.fit(
    X_train[no_iculos_cols],
    dev_A["labels"]
)

prob_A_no_iculos = model_no_iculos.predict_proba(
    X_test_A[no_iculos_cols]
)[:, 1]

prob_B_no_iculos = model_no_iculos.predict_proba(
    X_test_B[no_iculos_cols]
)[:, 1]

results_no_iculos = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_no_iculos, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_no_iculos, threshold=0.55
    )
}).T

print("Features:", len(no_iculos_cols))
print(results_no_iculos.round(4).to_string())

Features: 35
                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7114  0.0471  0.0864     0.0471  0.5239  0.2334            0.2196
Hospital B        0.6760  0.0368  0.0647     0.0350  0.4177  0.1650            0.0816


In [18]:
# Logistic Regression using ICULOS only
model_iculos = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_iculos.fit(
    X_train[["ICULOS"]],
    dev_A["labels"]
)

# Predict on both hospitals
prob_A_iculos = model_iculos.predict_proba(
    X_test_A[["ICULOS"]]
)[:, 1]

prob_B_iculos = model_iculos.predict_proba(
    X_test_B[["ICULOS"]]
)[:, 1]

# Evaluate at the original threshold
results_iculos = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_iculos, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_iculos, threshold=0.55
    )
}).T

print(results_iculos.round(4).to_string())

                  AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.666  0.0779  0.1590     0.0985  0.4127  0.0832            0.3164
Hospital B        0.628  0.0477  0.1072     0.0622  0.3888  0.0842            0.2325


In [19]:
# Remove ICULOS, but keep all missingness features
cols_no_iculos_missing = [
    c for c in X_train_flags_since.columns
    if c != "ICULOS"
]

model_no_iculos_missing = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

# Train on Hospital A development patients
model_no_iculos_missing.fit(
    X_train_flags_since[cols_no_iculos_missing],
    dev_A["labels"]
)

# Predictions
prob_A_no_iculos_missing = model_no_iculos_missing.predict_proba(
    X_test_A_flags_since[cols_no_iculos_missing]
)[:, 1]

prob_B_no_iculos_missing = model_no_iculos_missing.predict_proba(
    X_test_B_flags_since[cols_no_iculos_missing]
)[:, 1]

# Evaluate using the original cutoff
results_no_iculos_missing = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_no_iculos_missing, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_no_iculos_missing, threshold=0.55
    )
}).T

print("Features:", len(cols_no_iculos_missing))
print(results_no_iculos_missing.round(4).to_string())

Features: 95
                  AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.758  0.0610  0.0997     0.0541  0.6270  0.2410            0.3167
Hospital B        0.709  0.0419  0.0679     0.0367  0.4629  0.1745            0.1081


In [20]:
# Clinical measurements + their missingness features
clinical_missing_cols = (
    kept_clinical + flag_cols + since_cols
)

model_clinical_missing = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_clinical_missing.fit(
    X_train_flags_since[clinical_missing_cols],
    dev_A["labels"]
)

prob_A_clinical_missing = model_clinical_missing.predict_proba(
    X_test_A_flags_since[clinical_missing_cols]
)[:, 1]

prob_B_clinical_missing = model_clinical_missing.predict_proba(
    X_test_B_flags_since[clinical_missing_cols]
)[:, 1]

results_clinical_missing = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_clinical_missing, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_clinical_missing, threshold=0.55
    )
}).T

print("Features:", len(clinical_missing_cols))
print(results_clinical_missing.round(4).to_string())

Features: 90
                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7475  0.0618  0.0964     0.0524  0.6025  0.2397            0.2926
Hospital B        0.6981  0.0384  0.0664     0.0359  0.4314  0.1660            0.0957


In [21]:
# Same function from Notebook 07
def add_trends(data, features, window=3):
    df = data.sort_values(["PatientID", "Hour"]).copy()

    for feature in features:
        filled = df.groupby("PatientID")[feature].ffill()
        grouped = filled.groupby(df["PatientID"])

        df[f"{feature}_mean_{window}h"] = grouped.transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        ).astype("float32")

        df[f"{feature}_std_{window}h"] = grouped.transform(
            lambda x: x.rolling(window, min_periods=1).std(ddof=0)
        ).astype("float32")

        df[f"{feature}_change_{window}h"] = (
            filled - grouped.shift(window)
        ).astype("float32")

    return df


# Same 7 vital signs from our original experiments
vital_features = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]

trend_cols = [
    f"{feature}_{stat}_3h"
    for feature in vital_features
    for stat in ["mean", "std", "change"]
]

# Start with Hospital A development data
trend_input = dev_A["filled"][vital_features].copy()
trend_input["PatientID"] = dev_A["patients"]
trend_input["Hour"] = dev_A["hours"]

trend_data_A = add_trends(trend_input, vital_features)

# Verify row alignment
assert np.array_equal(trend_data_A["PatientID"], dev_A["patients"])
assert np.array_equal(trend_data_A["Hour"], dev_A["hours"])

print("Development rows:", len(trend_data_A))
print("New trend features:", len(trend_cols))

Development rows: 631120
New trend features: 21


In [22]:
# Hospital A test data
trend_input_test_A = test_A["filled"][vital_features].copy()
trend_input_test_A["PatientID"] = test_A["patients"]
trend_input_test_A["Hour"] = test_A["hours"]

trend_data_test_A = add_trends(trend_input_test_A, vital_features)

# Hospital B test data
trend_input_test_B = test_B["filled"][vital_features].copy()
trend_input_test_B["PatientID"] = test_B["patients"]
trend_input_test_B["Hour"] = test_B["hours"]

trend_data_test_B = add_trends(trend_input_test_B, vital_features)

# Verify alignment
assert np.array_equal(trend_data_test_A["PatientID"], test_A["patients"])
assert np.array_equal(trend_data_test_B["PatientID"], test_B["patients"])

print("Hospital A test trends:", trend_data_test_A[trend_cols].shape)
print("Hospital B test trends:", trend_data_test_B[trend_cols].shape)

Hospital A test trends: (159095, 21)
Hospital B test trends: (761995, 21)


In [23]:
# Start with our already-preprocessed 36 original features
X_train_vital = X_train.copy()
X_test_A_vital = X_test_A.copy()
X_test_B_vital = X_test_B.copy()

# Add the same 21 trends to each dataset
X_train_vital[trend_cols] = trend_data_A[trend_cols].to_numpy()
X_test_A_vital[trend_cols] = trend_data_test_A[trend_cols].to_numpy()
X_test_B_vital[trend_cols] = trend_data_test_B[trend_cols].to_numpy()

# Impute missing trend values using training medians only
trend_medians = X_train_vital[trend_cols].median()

for df in [X_train_vital, X_test_A_vital, X_test_B_vital]:
    df[trend_cols] = df[trend_cols].fillna(trend_medians)

# Standardize trends using Hospital A development statistics only
trend_scaler = StandardScaler()

X_train_vital[trend_cols] = trend_scaler.fit_transform(
    X_train_vital[trend_cols]
)

X_test_A_vital[trend_cols] = trend_scaler.transform(
    X_test_A_vital[trend_cols]
)

X_test_B_vital[trend_cols] = trend_scaler.transform(
    X_test_B_vital[trend_cols]
)

print("Training:", X_train_vital.shape)
print("Hospital A:", X_test_A_vital.shape)
print("Hospital B:", X_test_B_vital.shape)

print("Remaining missing values:",
      sum(df.isna().sum().sum()
          for df in [X_train_vital, X_test_A_vital, X_test_B_vital]))

Training: (631120, 57)
Hospital A: (159095, 57)
Hospital B: (761995, 57)
Remaining missing values: 0


In [24]:
# Weighted Logistic Regression with vital-sign trends
model_vital = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

# Train on Hospital A development data
model_vital.fit(X_train_vital, dev_A["labels"])

# Predict on both hospitals
prob_A_vital = model_vital.predict_proba(X_test_A_vital)[:, 1]
prob_B_vital = model_vital.predict_proba(X_test_B_vital)[:, 1]

# Evaluate using the original threshold
results_vital = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_vital, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_vital, threshold=0.55
    )
}).T

print(results_vital.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7635  0.0801  0.1148     0.0637  0.5794  0.1874            0.3435
Hospital B        0.7218  0.0549  0.0802     0.0436  0.4985  0.1568            0.1906


In [25]:
# Same laboratory features as Notebook 07
lab_features = list(s.LABS)

lab_trend_cols = [
    f"{feature}_{stat}_3h"
    for feature in lab_features
    for stat in ["mean", "std", "change"]
]

# Hospital A development data
lab_input_A = dev_A["filled"][lab_features].copy()
lab_input_A["PatientID"] = dev_A["patients"]
lab_input_A["Hour"] = dev_A["hours"]

# Reuse our original add_trends function
lab_trend_data_A = add_trends(lab_input_A, lab_features)

# Verify alignment
assert np.array_equal(
    lab_trend_data_A["PatientID"], dev_A["patients"]
)
assert np.array_equal(
    lab_trend_data_A["Hour"], dev_A["hours"]
)

print("Development lab trends:", lab_trend_data_A[lab_trend_cols].shape)
print("Laboratory trend features:", len(lab_trend_cols))

Development lab trends: (631120, 78)
Laboratory trend features: 78


In [26]:
# Hospital A test data
lab_input_test_A = test_A["filled"][lab_features].copy()
lab_input_test_A["PatientID"] = test_A["patients"]
lab_input_test_A["Hour"] = test_A["hours"]

lab_trend_data_test_A = add_trends(
    lab_input_test_A, lab_features
)

# Hospital B test data
lab_input_test_B = test_B["filled"][lab_features].copy()
lab_input_test_B["PatientID"] = test_B["patients"]
lab_input_test_B["Hour"] = test_B["hours"]

lab_trend_data_test_B = add_trends(
    lab_input_test_B, lab_features
)

# Verify alignment
assert np.array_equal(
    lab_trend_data_test_A["PatientID"], test_A["patients"]
)
assert np.array_equal(
    lab_trend_data_test_B["PatientID"], test_B["patients"]
)

print("Hospital A test:", lab_trend_data_test_A[lab_trend_cols].shape)
print("Hospital B:", lab_trend_data_test_B[lab_trend_cols].shape)

Hospital A test: (159095, 78)
Hospital B: (761995, 78)


In [27]:
# Check which lab trends meet the original missingness rule
lab_missing_pct = lab_trend_data_A[lab_trend_cols].isna().mean() * 100
lab_keep_cols = [c for c in lab_trend_cols if lab_missing_pct[c] < 99]

# Calculate medians from Hospital A development only
lab_medians = lab_trend_data_A[lab_keep_cols].median()

# Fill missing values and standardize
lab_scaler = StandardScaler()

train_lab = lab_scaler.fit_transform(
    lab_trend_data_A[lab_keep_cols].fillna(lab_medians).astype("float32")
)
test_A_lab = lab_scaler.transform(
    lab_trend_data_test_A[lab_keep_cols].fillna(lab_medians).astype("float32")
)
test_B_lab = lab_scaler.transform(
    lab_trend_data_test_B[lab_keep_cols].fillna(lab_medians).astype("float32")
)

# Combine with our previously prepared 57 features
X_train_vital_lab = pd.concat([
    X_train_vital,
    pd.DataFrame(train_lab, columns=lab_keep_cols)
], axis=1)

X_test_A_vital_lab = pd.concat([
    X_test_A_vital,
    pd.DataFrame(test_A_lab, columns=lab_keep_cols)
], axis=1)

X_test_B_vital_lab = pd.concat([
    X_test_B_vital,
    pd.DataFrame(test_B_lab, columns=lab_keep_cols)
], axis=1)

print("Lab trends retained:", len(lab_keep_cols))
print("Training:", X_train_vital_lab.shape)
print("Hospital A:", X_test_A_vital_lab.shape)
print("Hospital B:", X_test_B_vital_lab.shape)

Lab trends retained: 78
Training: (631120, 135)
Hospital A: (159095, 135)
Hospital B: (761995, 135)


In [28]:
# Experiment 2: Original features + vital trends + lab trends

model_vital_lab = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

# Train once on Hospital A development data
model_vital_lab.fit(X_train_vital_lab, dev_A["labels"])

# Predict on both hospitals
prob_A_vital_lab = model_vital_lab.predict_proba(
    X_test_A_vital_lab
)[:, 1]

prob_B_vital_lab = model_vital_lab.predict_proba(
    X_test_B_vital_lab
)[:, 1]

# Evaluate
results_vital_lab = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_vital_lab, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_vital_lab, threshold=0.55
    )
}).T

print(results_vital_lab.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7698  0.0800  0.1155     0.0642  0.5788  0.1858            0.3438
Hospital B        0.7177  0.0562  0.0869     0.0479  0.4691  0.1338            0.2070


In [29]:
# Experiment 3: Vital trends + missingness

extra_cols = flag_cols + since_cols

X3_train = pd.concat(
    [X_train_vital, X_train_flags_since[extra_cols]], axis=1
)
X3_A = pd.concat(
    [X_test_A_vital, X_test_A_flags_since[extra_cols]], axis=1
)
X3_B = pd.concat(
    [X_test_B_vital, X_test_B_flags_since[extra_cols]], axis=1
)

print("Features:", X3_train.shape[1])
assert X3_train.shape[1] == 117

model_exp3 = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_exp3.fit(X3_train, dev_A["labels"])

prob_A_exp3 = model_exp3.predict_proba(X3_A)[:, 1]
prob_B_exp3 = model_exp3.predict_proba(X3_B)[:, 1]

results_exp3 = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_exp3, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_exp3, threshold=0.55
    )
}).T

print(results_exp3.round(4).to_string())

Features: 117
                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7990  0.0908  0.1207     0.0668  0.6249  0.1922            0.3845
Hospital B        0.7457  0.0582  0.0951     0.0532  0.4506  0.1152            0.2252


In [30]:
# Experiment 4: Vital + lab trends + missingness

extra_cols = flag_cols + since_cols

X4_train = pd.concat(
    [X_train_vital_lab, X_train_flags_since[extra_cols]], axis=1
)
X4_A = pd.concat(
    [X_test_A_vital_lab, X_test_A_flags_since[extra_cols]], axis=1
)
X4_B = pd.concat(
    [X_test_B_vital_lab, X_test_B_flags_since[extra_cols]], axis=1
)

print("Features:", X4_train.shape[1])
assert X4_train.shape[1] == 195

# Train final model
model_exp4 = LogisticRegression(
    max_iter=1000,
    random_state=s.SEED,
    class_weight="balanced"
)

model_exp4.fit(X4_train, dev_A["labels"])

# Predict on both hospitals
prob_A_exp4 = model_exp4.predict_proba(X4_A)[:, 1]
prob_B_exp4 = model_exp4.predict_proba(X4_B)[:, 1]

# Evaluate at the original threshold
results_exp4 = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(
        test_A, prob_A_exp4, threshold=0.55
    ),
    "Hospital B": evaluate_test(
        test_B, prob_B_exp4, threshold=0.55
    )
}).T

print(results_exp4.round(4).to_string())

Features: 195
                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.7985  0.0908  0.1205     0.0668  0.6188  0.1904            0.3794
Hospital B        0.7272  0.0545  0.0911     0.0510  0.4236  0.1131            0.2035


In [31]:
# Collect all 14 completed Logistic Regression experiments
experiments = {
    "03 Weighted baseline": results,
    "03 Unweighted baseline": results_unweighted,
    "04 Measurement flags": results_flags,
    "04 Flags + hours-since": results_flags_since,
    "05 Clinical only": results_clinical,
    "05 Demographics + time": results_demo,
    "05 Without ICULOS": results_no_iculos,
    "06 ICULOS only": results_iculos,
    "06 Without ICULOS + missingness": results_no_iculos_missing,
    "06 Clinical + missingness": results_clinical_missing,
    "07 Vital trends": results_vital,
    "07 Vital + lab trends": results_vital_lab,
    "07 Vital trends + missingness": results_exp3,
    "07 Vital + lab trends + missingness": results_exp4
}

# Combine into one organized results table
summary = pd.concat(
    experiments,
    names=["Experiment", "Hospital"]
).reset_index()

display(summary.round(4))

# Export results
summary.to_csv("08_LR_Test_Results.csv", index=False)

from google.colab import files
files.download("08_LR_Test_Results.csv")

,Experiment,Hospital,AUROC,AUPRC,F1,Precision,Recall,FPR,Clinical utility
0,03 Weighted baseline,Hospital A (20%),0.7591,0.0793,0.1153,0.0640,0.5771,0.1856,0.3419
1,03 Weighted baseline,Hospital B,0.7232,0.0548,0.0868,0.0477,0.4822,0.1382,0.2126
2,03 Unweighted baseline,Hospital A (20%),0.7590,0.0803,0.1230,0.1359,0.1124,0.0157,0.0954
3,03 Unweighted baseline,Hospital B,0.7217,0.0553,0.1143,0.1108,0.1181,0.0136,0.0951
4,04 Measurement flags,Hospital A (20%),0.7747,0.0844,0.1166,0.0647,0.5884,0.1873,0.3540
5,04 Measurement flags,Hospital B,0.7139,0.0516,0.0818,0.0449,0.4545,0.1386,0.1796
6,04 Flags + hours-since,Hospital A (20%),0.7956,0.0902,0.1216,0.0672,0.6346,0.1937,0.3892
7,04 Flags + hours-since,Hospital B,0.7459,0.0575,0.0970,0.0545,0.4381,0.1090,0.2251
8,05 Clinical only,Hospital A (20%),0.6769,0.0425,0.0809,0.0442,0.4737,0.2252,0.1788
9,05 Clinical only,Hospital B,0.6828,0.0378,0.0579,0.0308,0.4992,0.2258,0.0410


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>